In [83]:
import pandas as pd
import numpy as np
import minio
import warnings
import json
warnings.filterwarnings("ignore")
import os
from dotenv import load_dotenv
import psycopg2
from sqlalchemy import create_engine

# Load data from MinIO

In [84]:
# Load data from MinIO
env_path = os.path.join("..", ".env")
load_dotenv(env_path, override=True)  # override=True untuk reload values

# Print untuk verify credentials loaded
print(f"✅ Environment loaded from: {env_path}")
print(f"📍 MinIO Endpoint: {os.getenv('MINIO_ENDPOINT')}")
print(f"🔑 Access Key: {os.getenv('MINIO_ACCESS_KEY')[:4]}***")

minio_client = minio.Minio(
    endpoint=os.getenv("MINIO_ENDPOINT"),
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    secure=False
)

def load_data_from_minio(bucket_name, object_name):
    """Load JSON data from MinIO bucket"""
    import json
    try:
        response = minio_client.get_object(bucket_name=bucket_name, object_name=object_name)
        content = response.read().decode('utf-8')
        data = json.loads(content)
        
        # Handle both single dict and list of dicts
        if isinstance(data, dict):
            df = pd.DataFrame([data])  # Wrap in list if single dict
        else:
            df = pd.DataFrame(data)
        
        return df
    except Exception as e:
        print(f"❌ Error accessing MinIO: {e}")
        raise

✅ Environment loaded from: ../.env
📍 MinIO Endpoint: localhost:9010
🔑 Access Key: mini***


# List Objects in Bucket

In [85]:
# List semua objects di bucket pharmacy-data
print("📂 Listing objects in 'pharmacy-data' bucket...\n")

try:
    objects = minio_client.list_objects("pharmacy-data", recursive=True)
    
    gold_files = []
    silver_files = []
    bronze_files = []
    
    for obj in objects:
        if obj.object_name.startswith("gold/") and obj.object_name.endswith(".json"):
            gold_files.append(obj.object_name)
        elif obj.object_name.startswith("silver/"):
            silver_files.append(obj.object_name)
        elif obj.object_name.startswith("bronze/"):
            bronze_files.append(obj.object_name)
    
    print(f"📊 Summary:")
    print(f"  - Bronze files: {len(bronze_files)}")
    print(f"  - Silver files: {len(silver_files)}")
    print(f"  - Gold files: {len(gold_files)}")
    
    print(f"\n📁 Latest Gold files (last 10):")
    for file in sorted(gold_files)[-10:]:
        print(f"  - {file}")
        
except Exception as e:
    print(f"❌ Error: {e}")

📂 Listing objects in 'pharmacy-data' bucket...

📊 Summary:
  - Bronze files: 51
  - Silver files: 34
  - Gold files: 249

📁 Latest Gold files (last 10):
  - gold/20260117_210431.json
  - gold/20260117_210432.json
  - gold/20260117_210433.json
  - gold/20260117_210434.json
  - gold/20260117_210435.json
  - gold/20260117_210436.json
  - gold/20260117_210437.json
  - gold/20260117_210438.json
  - gold/20260117_210439.json
  - gold/20260117_210440.json


In [86]:
pd.set_option('display.max_columns', None)

# Load gold data - gunakan file terbaru (format tanpa folder nested)
# File gold/ sekarang ada di format: gold/20260117_210424.json

try:
    # Coba load file terbaru
    df = load_data_from_minio(bucket_name="pharmacy-data", object_name="gold/20260117_210424.json")
    print(f"✅ Successfully loaded data from gold/20260117_210424.json")
    print(f"📊 Shape: {df.shape}")
    print(f"\n📋 Columns: {list(df.columns)}")
    print(f"\n🔍 Preview:")
    display(df.head())
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print(f"\n💡 Troubleshooting:")
    print("1. Make sure you've run the cell above to reload .env variables")
    print("2. Check credentials: MINIO_ACCESS_KEY and MINIO_SECRET_KEY in .env")
    print("3. Restart kernel and reload .env: load_dotenv(env_path)")

✅ Successfully loaded data from gold/20260117_210424.json
📊 Shape: (1, 4)

📋 Columns: ['product_name', 'year', 'month', 'total_sales']

🔍 Preview:


,product_name,year,month,total_sales
0,symbitrim,2018,January,5360.0


In [87]:
# Check what's inside the JSON file
import json

try:
    response = minio_client.get_object(bucket_name="pharmacy-data", object_name="gold/20260117_210424.json")
    content = response.read().decode('utf-8')
    
    print("📄 Raw content:")
    print(content[:500])  # First 500 chars
    
    print("\n\n🔍 Parsed as JSON:")
    data = json.loads(content)
    print(f"Type: {type(data)}")
    print(f"Content: {data}")
    
except Exception as e:
    print(f"❌ Error: {e}")

📄 Raw content:
{"product_name": "symbitrim", "year": 2018, "month": "January", "total_sales": 5360.0}


🔍 Parsed as JSON:
Type: <class 'dict'>
Content: {'product_name': 'symbitrim', 'year': 2018, 'month': 'January', 'total_sales': 5360.0}


In [88]:
# Simple Load Data from MinIO

In [89]:
# Simple function to load all gold data from MinIO
def load_all_gold_data():
    """Load all gold JSON files from MinIO and combine into single DataFrame"""
    all_data = []
    errors = []
    
    # List all objects in gold folder
    objects = minio_client.list_objects("pharmacy-data", prefix="gold/", recursive=True)
    
    for obj in objects:
        if obj.object_name.endswith(".json"):
            try:
                # Get and parse JSON
                response = minio_client.get_object("pharmacy-data", obj.object_name)
                data = json.loads(response.read().decode('utf-8'))
                
                # Add to list (handle both dict and list)
                if isinstance(data, dict):
                    all_data.append(data)
                elif isinstance(data, list):
                    all_data.extend(data)
            except Exception as e:
                errors.append((obj.object_name, str(e)))
    
    if errors:
        print(f"⚠️  Failed to load {len(errors)} files")
        for filename, error in errors[:5]:  # Show first 5 errors
            print(f"  - {filename}: {error[:50]}...")
    
    # Convert to DataFrame
    return pd.DataFrame(all_data)

# Load data
df = load_all_gold_data()

print(f"\n✅ Data loaded successfully!")
print(f"📊 Shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")
print(f"\n🔍 First 5 rows:")
df.head()

⚠️  Failed to load 92 files
  - gold/2018/April/aspinavir+silovance.json: S3 operation failed; code: NoSuchKey, message: The...
  - gold/2018/January/abilovir+aprotasol.json: S3 operation failed; code: NoSuchKey, message: The...
  - gold/2018/January/abranatal+lysoprosate.json: S3 operation failed; code: NoSuchKey, message: The...
  - gold/2018/January/acycnafine+microvate.json: S3 operation failed; code: NoSuchKey, message: The...
  - gold/2018/January/adrecetam+barazoxane.json: S3 operation failed; code: NoSuchKey, message: The...

✅ Data loaded successfully!
📊 Shape: (157, 4)
📋 Columns: ['product_name', 'year', 'month', 'total_sales']

🔍 First 5 rows:


,product_name,year,month,total_sales
0,abatatriptan,2018,January,8904.0
1,abobozolid,2018,January,7050.0
2,abtasol,2018,January,42224.0
3,acantaine,2018,January,21780.0
4,acelimus,2018,January,8952.0


# Alternative: One-liner Load (Super Simple)

In [96]:
# Super compact version with error handling
def safe_load_object(obj):
    """Helper to safely load a single JSON object"""
    try:
        response = minio_client.get_object("pharmacy-data", obj.object_name)
        return json.loads(response.read().decode('utf-8'))
    except:
        return None  # Skip error files

# Load all gold data (skip errors automatically)
df_simple = pd.DataFrame([
    data for data in [
        safe_load_object(obj) 
        for obj in minio_client.list_objects("pharmacy-data", prefix="gold", recursive=True)
        if obj.object_name.endswith(".json")
    ] if data is not None  # Filter out None (error) results
])

print(f"✅ Loaded {len(df_simple)} records")
df_simple

✅ Loaded 157 records


,product_name,year,month,total_sales
0,abatatriptan,2018,January,8904.0
1,abobozolid,2018,January,7050.0
2,abtasol,2018,January,42224.0
3,acantaine,2018,January,21780.0
4,acelimus,2018,January,8952.0
...,...,...,...,...
152,lacoplex hyaluvance,2018,January,3006.0
153,palodizem,2018,January,1407.0
154,exexone,2018,January,1195.0
155,araxetine,2018,January,2250.0


# Data Overview & Basic Statistics

In [91]:
# Data Info
print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\n📈 Missing values:\n{df.isnull().sum()}")
print(f"\n🔢 Statistical Summary:")
df.describe()

📊 DATASET OVERVIEW
Shape: 157 rows × 4 columns

Columns: ['product_name', 'year', 'month', 'total_sales']

Data types:
product_name     object
year              int64
month            object
total_sales     float64
dtype: object

📈 Missing values:
product_name    0
year            0
month           0
total_sales     0
dtype: int64

🔢 Statistical Summary:


,year,total_sales
count,157.0,157.000000
mean,2018.0,31841.980892
std,0.0,53244.635871
min,2018.0,82.000000
25%,2018.0,3888.000000
50%,2018.0,10890.000000
75%,2018.0,42228.000000
max,2018.0,469898.000000
